# Python SQL notebook with Almighty Oracle

_Version: 2025-01-13_

# First steps


## Requirements

- Connect to Univie VPN ([HowTo](https://moodle.univie.ac.at/mod/book/view.php?id=19086186&chapterid=154722))
- Python – [https://www.python.org/downloads/](https://www.python.org/downloads/)


## OracleDB module

Before we can connect we need to install a additonal package.

The `python-oracledb` module to connect to the database: `pip  install oracledb`.

Documentation can be found at [https://python-oracledb.readthedocs.io/en/latest/](https://python-oracledb.readthedocs.io/en/latest/).

## Let's start connecting!

Connect to the **UniVie VPN**.

In [ ]:
!pip  install oracledb

In [ ]:
import oracledb

The Oracle Instant Client comes prepackaged in the dbs_python.zip.
So we just need to call it to use python-oracledb in **Thick** mode.

In [ ]:
oracledb.init_oracle_client(lib_dir=r"instantclient_19_11")
# add here the path to the instantclient

### Set the credentials

The defaults are as follow:

- Username: [your username] (e.g. a01234567)  
- Password: [your password] (default-password: dbsXX where XX is the last two digits of the current year, e.g. 22)

**It is recommended to connect to Almighty, use `sqlplus`, log in and change the password with `passw`.**

In [ ]:
import os
from getpass import getpass

# Credentials are read from environment variables (see .env.example).
# Never hardcode credentials in the notebook!
username = os.environ.get("ORACLE_USER") or input("DB username: ")
password = os.environ.get("ORACLE_PASSWORD") or getpass("DB password: ")
con_string = os.environ.get("ORACLE_DSN", "oracle19.cs.univie.ac.at:1521/orclcdb")


### Connect

Documentation can be found at [https://python-oracledb.readthedocs.io/en/latest/user_guide/connection_handling.html](https://python-oracledb.readthedocs.io/en/latest/user_guide/connection_handling.html).

In `cx_Oracle` the paramaters `encoding` and `nencoding` were supported to be passed to `oracledb.connect()`.
This is no longer the case!

In [ ]:
#consider to use with in order to avoid open connections
connection = oracledb.connect(user=username,password=password,
                              dsn=con_string)
'''
with oracledb.connect(user=username,password=password, dsn=con_string) as connection:
    with connection.cursor() as cursor:
    ...
'''

Check if connection is now up!

In [ ]:
#create a table
curs=connection.cursor()
create_stat="create table myTestTable(myId INTEGER)" #just an example: always use a PK
curs.execute(create_stat)
connection.commit()

In [ ]:
#insert into
myIdValue=4
create_stat="insert into myTestTable (myId) VALUES (:myIdValue)"
curs.execute(create_stat,myIdValue=myIdValue)
connection.commit()

In [ ]:
#select
curs.execute("select * from myTestTable")
res = curs.fetchall()

In [ ]:
str(res)  #result should give '[(4,)]'

In [ ]:
[x[0] for x in res]

### Read from CSV

You need to install `pandas` from your CMD.
`python -m pip install pandas`

In [ ]:
!pip install pandas

In [ ]:
import pandas as pd

In [ ]:
example_data=pd.read_csv("ExampleData.csv",sep=";")

In [ ]:
example_data.head()

In [ ]:
#create a table
create_stat="create table myTestTable2(myId INTEGER,myName VARCHAR(30))" #just an example: always use a PK
curs.execute(create_stat)
connection.commit()

In [ ]:
def add_tuple(m_tuple):
    create_stat="insert into myTestTable2 (myId,myName) VALUES (:myId,:myName)"
    curs.execute(create_stat,myId=m_tuple.Id,myName=m_tuple.Name)
    connection.commit()

In [ ]:
example_data.apply(lambda x: add_tuple(x),axis=1)  #use for instead of apply if that is easier for your!
print("done with inserting....")

In [ ]:
curs.execute("select * from myTestTable2")
res = curs.fetchall()
res # result should give [(9, 'Ann'), (10, 'Bob'), (11, 'Carol')]

### Close Connection

In [ ]:
curs.close()
connection.close()

---------------------------------------

# Aufgabe 3-3

## Tabellen erstellen

In [ ]:
cursor = connection.cursor()


In [ ]:
with open("create_script.sql", "r", encoding="utf-8") as f:
    ddl_script = f.read()

for stmt in ddl_script.split(";"):
    stmt = stmt.strip()
    if not stmt:
        continue
    cursor.execute(stmt)

connection.commit()

print("fertig")


## Daten einfügen

In [ ]:
import random
import datetime
import oracledb

cursor = connection.cursor()

In [ ]:
# 150 Fahrzeuge
for _ in range(150):
    marke = random.choice(["VW","Audi","BMW","Mercedes","Opel"])
    modell = f"Typ{random.randint(100,999)}"
    jahr   = random.randint(2000, 2024)
    cursor.execute("""
        insert into Fahrzeug (Marke, Modell, Baujahr)
        values (:1, :2, :3)
    """, [marke, modell, jahr])


In [ ]:
# 2000 Telefonnummern
phones = set()
while len(phones) < 2000:
    num = f"+43{random.randint(100000000,999999999)}"
    try:
        cursor.execute("insert into Telefonnummer(Nummer) values(:1)", [num])
        phones.add(num)
    except oracledb.IntegrityError:
        pass
phones = list(phones)


In [ ]:
# 2000 Kunden
phone_list = list(phones)
random.shuffle(phone_list)

fsns = []
for i in range(2000):
    fsn  = f"K{i:05d}"
    name = f"Cust{i}"
    dob  = (
        datetime.date(1950,1,1)
        + datetime.timedelta(days=random.randint(0,25000)))
    num  = phone_list[i]

    cursor.execute("""
        insert into Kunde (Fuehrerscheinnummer, Name, Geburtsdatum, Nummer)
        values (:1, :2, :3, :4)
    """, [fsn, name, dob, num])
    fsns.append(fsn)


In [ ]:
from collections import defaultdict

res_count = defaultdict(int)

# 2000 Reservierungen
for _ in range(2000):
    fsn   = random.choice(fsns)
    res_count[fsn] += 1
    rnr   = res_count[fsn]
    start = datetime.date(2024,1,1) + datetime.timedelta(days=random.randint(0,365))
    ende  = start + datetime.timedelta(days=random.randint(1,10))

    cursor.execute("""
        insert into Reservierung (Fuehrerscheinnummer, Reservierungsnummer, Reservierungsdatum, EndeDatum)
        values (:1, :2, :3, :4)
    """, [fsn, rnr, start, ende])


In [ ]:
connection.commit()

for table in ["Fahrzeug","Kunde","Reservierung"]:
    cursor.execute(f"SELECT COUNT(*) FROM {table}")
    print(table, cursor.fetchone()[0])


## Queries

In [ ]:
cursor = connection.cursor()

# kunden mit > 3 reservierungen
cursor.execute("""
select r.fuehrerscheinnummer, count(*) as anzres
from reservierung r
group by r.fuehrerscheinnummer
having count(*) > 3
""")
print("kunden mit >3 reservierungen:", cursor.fetchall())


# kunden deren geburtsdatum nach allen 1970 liegt
cursor.execute("""
select k.name, k.fuehrerscheinnummer
from kunde k
where k.geburtsdatum > all (
    select date '1970-01-01' from dual
)
""")
print("kunden geburtsdatum > alle 1970:", cursor.fetchall())


# Fahrzeuge deren baujahr nach allen audis liegt
cursor.execute("""
select f.marke, f.modell, f.baujahr
from fahrzeug f
where f.baujahr > all (
    select baujahr
    from fahrzeug
    where marke = 'audi')
""")
print("fahrzeuge baujahr > alle audi:", cursor.fetchall())


## Stores Procudures

In [ ]:
cursor.execute("""
create or replace procedure sp_update_fahrzeug(
    p_id in number,
    p_neuesmod in varchar2
    ) as
begin
    update fahrzeug
    set modell = p_neuesmod
    where fahrzeug_id = p_id;
    commit;
end;
""")

cursor.execute("""
create or replace procedure sp_delete_reservierung(
    p_fsn in char,
    p_endmax in date
    ) as
begin
    delete from reservierung
    where fuehrerscheinnummer = p_fsn
    and endedatum <= p_endmax;
    commit;
end;
""")

connection.commit()
print("fertig")


In [ ]:
import datetime

cursor.callproc('sp_update_fahrzeug', [123, 'Golf GTI'])

end_datum = datetime.date(2024, 12, 31)
cursor.callproc('sp_delete_reservierung', ['FS1234567', end_datum])


## Index

In [ ]:
# von der angabe
cursor.execute("""
CREATE TABLE kunden (
    id NUMBER PRIMARY KEY,
    vorname VARCHAR2(50),
    nachname VARCHAR2(50),
    email VARCHAR2(100),
    geburtsdatum DATE,
    stadt VARCHAR2(50)
)
""")


In [ ]:
cursor.execute("CREATE SEQUENCE kunden_seq START WITH 1 INCREMENT BY 1")

In [ ]:
cursor.execute("""
BEGIN
    FOR i IN 1..100000 LOOP
        INSERT INTO kunden (
            id, vorname, nachname, email, geburtsdatum, stadt
        ) VALUES (
            kunden_seq.NEXTVAL,
            'Vorname' || i,
            'Nachname' || i,
            'email' || i || '@example.com',
            TO_DATE('1990-01-01', 'YYYY-MM-DD'),
            CASE 
                WHEN DBMS_RANDOM.VALUE < 0.5 THEN 'Wien' 
                ELSE 'Graz' 
            END
        );
    END LOOP;
    COMMIT;
END;
""")

In [ ]:
connection.commit()

In [ ]:
cursor.execute("select count(*) from kunden")
print(cursor.fetchall())

In [ ]:
import time

# explain plan
cursor.execute("""
explain plan for
select * from kunden where stadt = 'Wien'
""")

cursor.execute("""
select * from table(dbms_xplan.display())
""")
for row in cursor:
    print(row)

start = time.time()

cursor.execute("""
select * from kunden where stadt = 'Wien'
""")
rows = cursor.fetchall()

end = time.time()
print(f"Anzahl der zeilen: {len(rows)}")
print(f"Ausführungszeit ohne index: {end - start:.4f} sekunden")


In [ ]:
cursor.execute("""
create index idx_stadt on kunden (stadt)
""")

connection.commit()
print("Index erstellt")


In [ ]:
# explain plan mit index
cursor.execute("""
explain plan for
select * from kunden where stadt = 'Wien'
""")

cursor.execute("""
select * from table(dbms_xplan.display())
""")
for row in cursor:
    print(row)

start = time.time()

cursor.execute("""
select * from kunden where stadt = 'Wien'
""")
rows = cursor.fetchall()

end = time.time()
print(f"Anzahl der zeilen: {len(rows)}")
print(f"Ausführungszeit mit index: {end - start:.4f} sekunden")


In [ ]:
cursor.execute("""
drop index idx_stadt
               """)
connection.commit()

**Vor dem index: 8,8 sekunden**

**Nach dem index: 8,8 sekunden**

(wahrscheinlich wurde der Index gar nicht verwendet)